# SBERT Sentence Analysis

## 1. Preparations

### 1.1 Read Data
This new data is obtained after some further anonymization, and by putting previously incorrectly split sentences together, and removed duplicates

In [1]:
import pandas as pd
# read the sentence data
df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_data/sentence_data_new.xlsx", index_col=0)
# check the head
df.head()

,Message_ID,Sentence,Clean_Sentence,Translated_Sentence,Sentence_ID
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,I was rushed into the [PRESSION] two weeks ago...,1
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",I was acutely under a painful pressure on the ...,2
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,It started 1 hour after dinner.,3
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...","I had migraines all day, so I hadn't eaten much.",4
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...","I got very nauseous, vomiting, dizzy and rejuv...",5


In [2]:
# clean sentences as inputs
sentences = df["Clean_Sentence"].to_list()

In [ ]:
# print the size of the data
len(df)
# There are 39408 unique sentences 

### 1.2. Import the list of stopwords

In [3]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/data/analysis_data/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec', "jl", "weken", "week", "dagen", "dag", "mg", "coach", "mijnibdcoach", "dr", "uur", "dgs"
]

dutch_stopwords.extend(extra_list)

### 1.4. Embed the lists of sentences with sentence-transformer multilingual-v1

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# load the embedding model
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")

# run the following for embeddings if no pre-saved embeddings can be loaded
#embeddings = embedding_model.encode(sentences, show_progress_bar=True)
# save the pre-calculated embeddings 
#np.save('/workspace/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy', embeddings)

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
# load pre-saved embeddings if you have, otherwise calculate them using the commented-out codes
embedding_model = SentenceTransformer("distiluse-base-multilingual-cased-v1")
embeddings = np.load("/workspace/mijnidbcoachnlp/data/embeddings/embeddings_st_v1_new_input.npy")

/workspace/mijnidbcoachnlp/venv/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2025-02-08 23:49:53.396989: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### 1.5 functions to pre-configure burtopic model

In [5]:
# set the vectorizer model
from sklearn.feature_extraction.text import CountVectorizer

# configure the sub-models
#vectorizer
vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b')

In [6]:
# function to build the model with different selection of clustering method
# build the BERTopic model
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

def tune_umap(n_neighbors, n_components, min_dist):
    umap_model = UMAP(n_neighbors=n_neighbors, n_components=n_components, min_dist=min_dist, metric='cosine', random_state=42)
    return umap_model

def tune_hdb(min_cluster_size, min_samples, cluster_selection_epsilon):
    hdb_model = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, cluster_selection_epsilon=cluster_selection_epsilon, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
    return hdb_model

def build_bertopic(embedding_model, umap_model, cluster_model, vectorizer_model, embeddings, docs):
    # Initialize BERTopic with UMAP and HDBSCAN models
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        verbose=True
    )

    # Fit-transform to get document topics and probabilities
    topics, probs = topic_model.fit_transform(docs, embeddings)
    

    # Return only essential results
    return topic_model

## 2. Build BERTopic Model

In [ ]:
# disable parallelism to avoid some warnings
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# build the model with the preconfigured settings in 1.5
# use a empirical set of parameters 
umap_model = tune_umap(15, 5, 0.05)
hdb_model = tune_hdb(20, 10, 0.0)
topic_model = build_bertopic(embedding_model=embedding_model, umap_model=umap_model, cluster_model=hdb_model, vectorizer_model=vectorizer_model, embeddings=embeddings, docs=sentences)

In [ ]:
# examine topic information
pd.set_option("max_colwidth", 200) # adjust column width 
topic_model.get_topic_info()

In [7]:
# Calculate the coherence score of the model
# Note: to evaluate with coherence score, the n_gram of the vectorizer_model has to be (1, 1)
def get_top_words(topic_model):
    topics = topic_model.get_topics()
    top_words = [[word for word, _ in topic[:10]] for topic in topics.values() if topic]  # Top 10 words per topic
    return top_words

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

# Create a dictionary representation of the words in topics
tokenized_texts = [doc.split() for doc in sentences]  # Tokenizing by splitting on spaces
dictionary = Dictionary(tokenized_texts)

# Create a coherence model
coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')

# Compute coherence score
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score: {coherence_score}")


In [ ]:
topic_coherences = coherence_model.get_coherence_per_topic()
print(topic_coherences)

## 3. Fine-Tune the Model

In [8]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

tokenized_texts = [doc.split() for doc in sentences]  # Tokenizing by splitting on spaces
dictionary = Dictionary(tokenized_texts)


def calculate_c_v(topic_model):
    top_words = get_top_words(topic_model)
    coherence_model = CoherenceModel(topics=top_words, 
                                texts=tokenized_texts,
                                dictionary=dictionary, 
                                coherence='c_v')
    coherence_score = coherence_model.get_coherence()
    return coherence_score

In [ ]:
import itertools

# Define parameter ranges
range_n_neighbors = [5, 10, 20, 35, 50, 75, 100]
range_n_components = [2, 3, 4, 5, 6]
range_min_dist = [0.0, 0.01, 0.05, 0.1]

# Generate all possible combinations of the parameters
umap_combinations = list(itertools.product(range_n_neighbors, range_n_components, range_min_dist))

In [9]:
import random
from joblib import Parallel, delayed

def tune_bertopic_umap_random(embedding_model, vectorizer_model, embeddings, sentences, n_iter=10):
    """
    Randomized search over UMAP and HDBSCAN hyperparameters to find the best model based on coherence score.
    
    Args:
        embedding_model: The embedding model to use for topic modeling.
        vectorizer_model: The vectorizer model to use for tokenization.
        embeddings: The embeddings of the sentences.
        sentences: The sentences (documents) for topic modeling.
        n_iter: The number of random iterations to perform in the randomized search.
        
    Returns:
        best_topic_model: The best topic model based on coherence score.
        best_coherence_score: The best coherence score obtained during the search.
    """
    
    # Define ranges for the UMAP parameters
    range_n_neighbors = [5, 10, 20, 35, 50, 75, 100]
    range_n_components = [2, 3, 4, 5, 6]
    range_min_dist = [0.0, 0.01, 0.05, 0.1]
    
    best_coherence_score = -float('inf')
    best_topic_model = None
    n_iter = 10
    # Randomized search loop
    for _ in range(n_iter):
        # Sample random UMAP parameters
        n_neighbors = random.choice(range_n_neighbors)
        n_components = random.choice(range_n_components)
        min_dist = random.choice(range_min_dist)

        # Tune UMAP and HDBSCAN
        umap_model = tune_umap(n_neighbors=n_neighbors, n_components=n_components, min_dist=min_dist)
        hdb_model = tune_hdb(min_cluster_size=20, min_samples=10, cluster_selection_epsilon=0.0)
        
        # Build topic model
        current_topic_model = build_bertopic(
            embedding_model=embedding_model, 
            umap_model=umap_model, 
            cluster_model=hdb_model, 
            vectorizer_model=vectorizer_model, 
            embeddings=embeddings, 
            docs=sentences
        )
        
        # Calculate coherence score for the current model
        current_coherence_score = calculate_c_v(current_topic_model)
        print(f"Coherence score: {current_coherence_score} for UMAP params: {n_neighbors}, {n_components}, {min_dist}")
        
        # Check if the current model has a better coherence score
        if current_coherence_score > best_coherence_score:
            best_coherence_score = current_coherence_score
            best_topic_model = current_topic_model
            best_combination = (n_neighbors, n_components, min_dist)
            
    return best_topic_model, best_coherence_score, best_combination

# Example usage
best_model, best_score, best_combination = tune_bertopic_umap_random(embedding_model, vectorizer_model, embeddings, sentences, n_iter=20)
print("Best UMAP Parameters:", best_combination)
print("Best Coherence Score:", best_score)


2025-02-08 23:50:22,506 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-08 23:51:51,664 - BERTopic - Dimensionality - Completed ✓
2025-02-08 23:51:51,667 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-08 23:51:55,486 - BERTopic - Cluster - Completed ✓
2025-02-08 23:51:55,501 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-08 23:51:56,280 - BERTopic - Representation - Completed ✓
2025-02-08 23:52:34,924 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.34745224566715555 for UMAP params: 20, 5, 0.0


2025-02-08 23:54:26,908 - BERTopic - Dimensionality - Completed ✓
2025-02-08 23:54:26,911 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-08 23:54:30,051 - BERTopic - Cluster - Completed ✓
2025-02-08 23:54:30,064 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-08 23:54:30,757 - BERTopic - Representation - Completed ✓
2025-02-08 23:55:02,939 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.3445548425803904 for UMAP params: 50, 6, 0.0


2025-02-08 23:55:44,153 - BERTopic - Dimensionality - Completed ✓
2025-02-08 23:55:44,156 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-08 23:55:46,071 - BERTopic - Cluster - Completed ✓
2025-02-08 23:55:46,084 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-08 23:55:46,879 - BERTopic - Representation - Completed ✓
2025-02-08 23:56:31,318 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


...

Coherence score: 0.35061016810592077 for UMAP params: 75, 3, 0.1


2025-02-09 00:15:09,646 - BERTopic - Dimensionality - Completed ✓
2025-02-09 00:15:09,650 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 00:15:12,932 - BERTopic - Cluster - Completed ✓
2025-02-09 00:15:12,949 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 00:15:13,862 - BERTopic - Representation - Completed ✓
2025-02-09 00:16:08,553 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Coherence score: 0.35136480046486684 for UMAP params: 20, 5, 0.01


2025-02-09 00:16:51,500 - BERTopic - Dimensionality - Completed ✓
2025-02-09 00:16:51,504 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-09 00:16:54,438 - BERTopic - Cluster - Completed ✓
2025-02-09 00:16:54,458 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-09 00:16:55,502 - BERTopic - Representation - Completed ✓


Coherence score: 0.347577988576024 for UMAP params: 5, 4, 0.1
Best UMAP Parameters: (75, 6, 0.05)
Best Coherence Score: 0.3554705572319291


## 3. Visualization and Merge Topics with a Threshold

In [ ]:
# settings for plotly
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default = "browser" # option to open the plots in brower

### 3.1 Visualize the hierarchy before topic merge

In [ ]:
hierarchical_topics = topic_model.hierarchical_topics(sentences)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

### 3.2 Merge topics with a threshold

## 4. Merge topics

### 4.1 Merge the topics of the model before after reduction

In [ ]:
import ast
# run this function to import the list of topic mappings that were used specifically for the saved ST model after topic reduction
topic_mapping_df = pd.read_csv("/workspace/mijnidbcoachnlp/data/result_data/mappings/topic_st_merge_mapping.csv", index_col=0) # read the saved mappings
topics_to_merge = topic_mapping_df["Topic_doc_info_group"].tolist() # convert to a list
topics_to_merge = [ast.literal_eval(item) for item in topics_to_merge] # convert to a format easy to copy-paste
for sublist in topics_to_merge:
    print(str(sublist)+",")
# Copy paste the list to the cell below


In [ ]:
# The clusters are eventually reduced 16 clusters that correspond to the 16 pre-defined labels
topics_to_merge = [[10, 11, 21, 23, 25, 39, 42, 51, 53, 55, 62, 63, 69, 76, 82, 84, 85, 86, 89, 92, 94, 95, 98, 102, 103, 105, 109, 110, 111, 120, 122, 124, 127, 128, 129, 130, 131, 135, 137, 141, 145, 146, 148, 153, 157, 159, 160, 161, 165, 171, 179, 187, 188, 195, 198, 201, 205, 208, 212, 214, 215, 216, 219, 222, 226, 227, 228, 229, 233, 234, 235, 237, 242, 245, 246, 247, 248, 249, 250, 252, 258, 263, 265, 266, 267, 268, 269, 270, 273, 279, 282, 284, 285, 287, -1],
[0, 2, 163, 231, 169, 75, 44, 142, 46, 80, 50, 243, 158, 223],
[1, 132, 134, 140, 14, 16, 147, 149, 278, 280, 283, 31, 33, 164, 167, 177, 180, 56, 185, 186, 74, 78, 79, 206, 83, 218, 96, 106, 125, 254],
[3, 264, 139, 271, 274, 24, 26, 27, 35, 168, 170, 45, 173, 178, 58, 59, 191, 65, 68, 87, 88, 91, 220, 221, 224, 97, 101, 112, 115, 118, 119, 121],
[256, 5, 133, 262, 261, 13, 272, 275, 276, 277, 28, 156, 286, 288, 43, 174, 176, 48, 52, 61, 189, 196, 70, 71, 72, 202, 203, 204, 230, 107, 108, 244, 126],
[7, 8, 73, 41, 12, 175, 18, 211, 20, 54, 182, 29, 190],
[64, 162, 194, 6, 166, 200, 199, 15, 143, 207, 117, 22, 151, 184, 281],
[138, 144, 155, 32, 34, 47, 57, 60, 192, 66, 67, 209, 210, 213, 217, 100, 238, 114, 251],
[4, 172, 81, 49, 17, 30],
[257, 99, 259, 37, 197, 90, 236, 77, 239, 150, 253, 154, 93, 255],
[193, 225, 38, 240, 113, 116, 183],
[136, 9, 104, 181],
[36, 260, 40, 232, 123],
[19],
[152],
[241]]

In [ ]:
topic_model = loaded_topic_model_ro # make sure again that the topic model is the one you want to merge
topic_model.merge_topics(sentences, topics_to_merge)

In [ ]:
topic_model.update_topics(sentences, vectorizer_model=vectorizer_model) # update the topic representation again 

In [ ]:
merged_topic_info = topic_model.get_topic_info()
merged_topic_info # this should have 16 topics now

### 4.2 Manually map the reduced clusters to pre-defined labels

In [ ]:
# create two new columns in the merged_topic_info df
merged_topic_info["Label"] = None
merged_topic_info["Category"] = None


In [ ]:
# save the topic and doc info after merging topics
merged_topic_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/merged_topic_info_st.xlsx', index=False)

Open the excel file, manually edit and "Label" column by putting the label codes A1-A5 and M1-M9"

In [ ]:
# read the saved merged_topic_info if necessary
#loaded_merged_topic_info = pd.read_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/merged_topic_info_st.xlsx')
#merged_topic_info = loaded_merged_topic_info
loaded_merged_topic_info.head()

In [ ]:
# you can also label it in the list here
merged_topic_info["Label"] = ["A5", "A4", "A2", "M8", "M6", "M3", "A3", "A1", "M2", "M4", "M5", "M10", "M1", "M7", "M11", "M9"]
# Map the labels to general catgories A/M
labeled_topic_info = merged_topic_info
labeled_topic_info.head() # check if the labels were put in

In [ ]:
# The label codes correspond to the following labels:
topic_names = ["Other", "Test Procedure & Results", "Appointment/Contact", "Symptoms", "Medication", "General Health Update",
                "Pharmacy", "Administrative Communication", "Consultation/Hospital Visit", "Medical Advice/Discussion", "Medical Examination",
                "Vaccination & Immunity", "Diet & Weight", "Pregnancy", "Work-Related Disease Management", "Surgery/Operation"]

In [ ]:
# automatically fill in the category A/M 
labeled_topic_info.loc[labeled_topic_info["Label"].str.contains("A"), "Category"] = "A"
labeled_topic_info.loc[labeled_topic_info["Label"].str.contains("M"), "Category"] = "M"
labeled_topic_info["Label_Name"] = topic_names

labeled_topic_info.head() # check the labeled topic_info again

In [ ]:
# save the labeled topic info
labeled_topic_info.to_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/merged_topic_info_st_labeled.xlsx', index=False)

## 5. Visualize topic frequency

In [ ]:
# read the data
import pandas as pd

labeled_topic_info = pd.read_excel('/workspace/mijnidbcoachnlp/data/result_data/topic_and_doc_info/merged_topic_info_st_labeled.xlsx')
labeled_topic_info

In [ ]:
import os

# Set the environment variable
os.environ["TOKENIZERS_PARALLELISM"] = "false"

### 5.1 Sentence topic frequency

In [ ]:
# plot as bar chart # This is the graph we used for ECCO abstract
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Select the relevant columns
data = labeled_topic_info[["Count", "Label_Name"]][1:]

# Sort the DataFrame by "Count" in descending order for better visualization
data = data.sort_values(by="Count", ascending=False).reset_index(drop=True)

# Normalize the counts to create a gradient effect
normalized_counts = (data["Count"] - data["Count"].min()) / (data["Count"].max() - data["Count"].min())

# Define two colormaps: Blue for administrative labels, Green for medical labels
blue_colors = plt.cm.Blues(normalized_counts)  # Administrative labels
green_colors = plt.cm.Greens(normalized_counts)  # Medical labels

# Specify which labels are administrative (blue)
administrative_labels = ["Test Procedure & Results", "Appointment/Contact", 
                         "General Health Update", "Pharmacy", "Administrative Communication"]

# Dynamically assign colors
colors = [blue_colors[i] if label in administrative_labels else green_colors[i] 
          for i, label in enumerate(data["Label_Name"])]

# Create the bar chart
plt.figure(figsize=(14, 9))
bars = plt.bar(data["Label_Name"], data["Count"], color=colors, edgecolor="black")

# Add count labels above the bars with an extended distance
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 20, int(yval), ha="center", fontsize=10)

# Add titles and labels
plt.title("Frequency of Topics", fontsize=16)
plt.xlabel("Topics", fontsize=12)
plt.ylabel("Count", fontsize=12)

# Add a caption below the x-axis label
plt.figtext(0.5, -0.05, "Figure 1. Frequency of Sentence-Level Topics", 
            wrap=True, horizontalalignment="center", fontsize=14)

# Move the custom legend slightly to the left
plt.gca().add_patch(mpatches.Rectangle((0.8, 0.8), 0.02, 0.02, color="blue", transform=plt.gcf().transFigure, clip_on=False))
plt.figtext(0.83, 0.8, "Administrative Labels", fontsize=12, color="black", ha="left")

plt.gca().add_patch(mpatches.Rectangle((0.8, 0.75), 0.02, 0.02, color="green", transform=plt.gcf().transFigure, clip_on=False))
plt.figtext(0.83, 0.75, "Medical Labels", fontsize=12, color="black", ha="left")

# Rotate x-axis labels for readability
plt.xticks(rotation=45, fontsize=11, ha="right")

# Adjust layout to avoid clipping
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Example DataFrame (replace with your actual data)

df = labeled_topic_info[["Count", "Label_Name", "Category"]][1:]

# Pivot the data so that we can have 'A' and 'M' in separate columns for each label
pivot_df = df.pivot(index='Label_Name', columns='Category', values='Count').fillna(0)

# Add a total count column, which is the sum of counts for each label across both categories
pivot_df['Total'] = pivot_df['A'] + pivot_df['M']

# Sort by the 'Total' column in descending order (highest to lowest)
pivot_df = pivot_df.sort_values(by='Total')

# Create a figure and axes
fig, ax = plt.subplots(figsize=(10, 6))

# Plot Category M on the left side of the plot (negative values for left-side bars)
ax.barh(pivot_df.index, pivot_df['M'], color='#A8D5BA', label='Medical', height=0.6)  # Softer green

# Plot Category A on the right side of the plot (positive values for right-side bars)
ax.barh(pivot_df.index, pivot_df['A'], color='#A7C7E7', label='Administrative', height=0.6)  # Softer blue

# Add labels and title
ax.set_xlabel('Count')
ax.set_title('Tornado Plot of Counts by Category (Ranked by Total Count)')
ax.legend()

# Show the plot
plt.tight_layout()
plt.show()


## 6. Compare computer labels to manual labels

### 6.1 Comparison using the small annotated set 
These are results we used for ECCO abstract

In [ ]:
# check out the document info of the current model after topic merge and 
merged_doc_info = topic_model.get_document_info(sentences)
merged_doc_info.head()

In [ ]:
# Save the document index as a new column "Sentence_ID" for concating with the annotated data
merged_doc_info["Sentence_ID"] = merged_doc_info.index

In [ ]:
# make sure the data type is the same of the "sentence_ID" columns of the two dataframes
annotated_df_sm['Sentence_ID'] = annotated_df_sm['Sentence_ID'].astype(str)
merged_doc_info['Sentence_ID'] = merged_doc_info['Sentence_ID'].astype(str)

In [ ]:
# Now perform the merge 
annotated_doc_sm = pd.merge( # merging the small annotated set with the document info
    annotated_df_sm,
    merged_doc_info[['Sentence_ID', 'Topic', 'Top_n_words']], # notice that we only choose these 3 columns of the doc info df
    on='Sentence_ID',
    how='left'
)

In [ ]:
annotated_df_sm.head(5)

In [ ]:
labeled_topic_info.head()

In [ ]:
# now merge the above dataframe with the labeled topic info frame
annotated_doc_sm = pd.merge(
    annotated_doc_sm,
    labeled_topic_info[['Topic', 'Label', 'Category', 'Label_Name']],
    on='Topic',
    how='left'
)

In [ ]:
annotated_doc_sm.head() # examine the annotated set after combining it with the computer-labels

In [ ]:
# Also add the category of the manual label
annotated_doc_sm["Manual_Category"] = None
annotated_doc_sm.loc[annotated_doc_sm["Manual_Label_Jiaxu"].str.contains("A"), "Manual_Category"] = "A"
annotated_doc_sm.loc[annotated_doc_sm["Manual_Label_Jiaxu"].str.contains("M"), "Manual_Category"] = "M"

In [ ]:
annotated_doc_sm.head()

In [ ]:
# import the annotated_doc_sm if it has been saved
#annotated_doc_sm = pd.read_excel("/workspace/mijnidbcoachnlp/data/result_data/Label_comparison/label_comparison_set_small.xlsx", index_col=0)
accuracy_results_sm = {}

for label in ["A5", "A4", "A2", "M8", "M6", "M3", "A3", "A1", "M2", "M4", "M5", "M10", "M1", "M7", "M11", "M9"]:
    # Filter the annotated document by the current label
    subset = annotated_doc_sm[annotated_doc_sm["Manual_Label_Jiaxu"] == label]
    
    # Check if there is a sufficient sample size
    if len(subset) > 10:
        # Calculate the accuracy as the percentage of matches in the filtered subset
        accuracy = (subset['Manual_Label_Jiaxu'] == subset['Label']).sum() / len(subset)
        accuracy_results_sm[label] = accuracy
        # Output the accuracy
        print(f"Accuracy for Manual_Label == {label}: {accuracy * 100:.2f}%")
    else:
        print(f"Too small sample size for label {label}")


In [ ]:
# Pring the average accuracy
accuracy_avg = (annotated_doc_sm['Manual_Label_Jiaxu'] == annotated_doc_sm['Label']).sum() / len(annotated_doc_sm)
print(f"Average Accuracy for Manual_Label: {accuracy_avg * 100:.2f}%")

In [ ]:
category_accuracy = (annotated_doc_sm['Manual_Category'] == annotated_doc_sm['Category']).sum() / len(annotated_doc_sm)
print(f"Accuracy for Category Matching: {category_accuracy * 100:.2f}%")

In [ ]:
# save the annotated_doc_sm
annotated_doc_sm.to_excel("/workspace/mijnidbcoachnlp/data/result_data/Label_comparison/label_comparison_set_small.xlsx")

### 6.2 Comparison with big annotated set

In [ ]:
annotated_df_lg['Sentence_ID'] = annotated_df_lg['Sentence_ID'].astype(str)

In [ ]:
annotated_df_lg.head()

In [ ]:
# Now perform the merge 
annotated_doc_lg = pd.merge( # merging the large annotated set with the document info
    annotated_df_lg,
    merged_doc_info[['Sentence_ID', 'Topic', 'Top_n_words']], # notice that we only choose these 3 columns of the doc info df
    on='Sentence_ID',
    how='left'
)

In [ ]:
annotated_doc_lg = pd.merge(
    annotated_doc_lg,
    labeled_topic_info[['Topic', 'Label', 'Category', 'Label_Name']],
    on='Topic',
    how='left'
)

In [ ]:
annotated_doc_lg.head() # examine the annotated set after combining it with the computer-labelsannotated_doc_sm

In [ ]:
# Also add the category of the manual label
annotated_doc_lg["Manual_Category"] = None
annotated_doc_lg.loc[annotated_doc_lg["Manual_label_Tom"].str.contains("A"), "Manual_Category"] = "A"
annotated_doc_lg.loc[annotated_doc_lg["Manual_label_Tom"].str.contains("M"), "Manual_Category"] = "M"

In [ ]:
# import the annotated_doc_sm if it has been saved
#annotated_doc_sm = pd.read_excel("/workspace/mijnidbcoachnlp/data/result_data/Label_comparison/label_comparison_set_small.xlsx", index_col=0)
accuracy_results_lg = {}

for label in ["A5", "A4", "A2", "M8", "M6", "M3", "A3", "A1", "M2", "M4", "M5", "M10", "M1", "M7", "M11", "M9"]:
    # Filter the annotated document by the current label
    subset = annotated_doc_lg[annotated_doc_lg["Manual_label_Tom"] == label]
    
    # Check if there is a sufficient sample size
    if len(subset) > 10:
        # Calculate the accuracy as the percentage of matches in the filtered subset
        accuracy = (subset['Manual_label_Tom'] == subset['Label']).sum() / len(subset)
        accuracy_results_lg[label] = accuracy
        # Output the accuracy
        print(f"Accuracy for Manual_Label == {label}: {accuracy * 100:.2f}%")
    else:
        print(f"Too small sample size for label {label}")


In [ ]:
# Pring the average accuracy
accuracy_avg_lg = (annotated_doc_lg['Manual_label_Tom'] == annotated_doc_lg['Label']).sum() / len(annotated_doc_lg)
print(f"Average Accuracy for Manual_Label: {accuracy_avg_lg * 100:.2f}%")

In [ ]:
category_accuracy_lg = (annotated_doc_lg['Manual_Category'] == annotated_doc_lg['Category']).sum() / len(annotated_doc_lg)
print(f"Accuracy for Category Matching: {category_accuracy_lg * 100:.2f}%")